## Example 1: Basic Email Validation and Normalization

In [ ]:
# Example 1: Basic Email Validation and Normalization

import email_validator
from email_validator import validate_email, EmailNotValidError

# Demonstrate basic email validation and normalization
print("=== Email Validation and Normalization ===")

try:
    # List of test email addresses
    test_emails = [
        "user@example.com",
        "User.Name+tag@EXAMPLE.COM",
        "user.name@sub.example.org",
        "firstname.lastname@company.co.uk",
        "test.email@domain-with-dash.com",
        "simple@domain.io",
        "user123@test123.net"
    ]
    
    print("=== Valid Email Validation ===")
    valid_results = []
    
    # check_deliverability=False throughout this cell: it demonstrates SYNTAX + normalization, and deliverability is a
    # live DNS lookup (measured: with network, example.com is rejected for its null MX; offline every address validates)
    for email in test_emails:
        # Validate and normalize email (every address in the list is valid: an EmailNotValidError fails the cell)
        validation_result = validate_email(email, check_deliverability=False)

        result_info = {
            "original": email,
            "normalized": validation_result.email,
            "local": validation_result.local_part,
            "domain": validation_result.domain,
            "ascii_email": validation_result.ascii_email,
            "smtputf8": validation_result.smtputf8
        }
        # measured (email-validator 2.3.0): the local part is kept as written, the domain is lower-cased
        local, domain = email.split("@")
        assert (validation_result.local_part, validation_result.domain) == (local, domain.lower()), result_info

        valid_results.append(result_info)

        print(f"✓ {email}")
        print(f"  Normalized: {validation_result.email}")
        print(f"  Local part: {validation_result.local_part}")
        print(f"  Domain: {validation_result.domain}")
        print(f"  ASCII form: {validation_result.ascii_email}")

    assert len(valid_results) == len(test_emails), valid_results
    
    print(f"\nValidation Summary:")
    print(f"  Total tested: {len(test_emails)}")
    print(f"  Valid: {len(valid_results)}")
    print(f"  Success rate: {len(valid_results)/len(test_emails):.1%}")
    
    # Show normalization effects
    print(f"\n=== Normalization Effects ===")
    # (input, its normalized form, or None where the library REJECTS it)
    normalization_examples = [
        ("User.Name@EXAMPLE.COM", "User.Name@example.com"),
        ("  spaced.email@domain.com  ", None),  # measured (2.3.0): surrounding spaces are a syntax error, not stripped
        ("Mixed.Case.Email@DOMAIN.ORG", "Mixed.Case.Email@domain.org")
    ]

    for email, expected in normalization_examples:
        try:
            result = validate_email(email, check_deliverability=False)
        except EmailNotValidError as e:
            assert expected is None and isinstance(e, email_validator.EmailSyntaxError) and "SPACE" in str(e), f"{email!r}: {e!r}"
            print(f"Rejected as expected: '{email}' - {e}")
        else:
            assert result.email == expected, f"{email!r} normalized to {result.email!r}, expected {expected!r}"
            print(f"Original: '{email}'")
            print(f"Normalized: '{result.email}'")
            print()
    
    # Demonstrate validation components
    print(f"=== Validation Components ===")
    sample_email = "test.user@example-domain.com"
    
    # a valid address: an EmailNotValidError here fails the cell
    result = validate_email(sample_email, check_deliverability=False)
    assert (result.local_part, result.domain, result.email, result.ascii_email, result.smtputf8) == (
        "test.user", "example-domain.com", sample_email, sample_email, False
    ), result
    print(f"Email: {sample_email}")
    print(f"Components:")
    print(f"  Local part: '{result.local_part}'")
    print(f"  Domain part: '{result.domain}'")
    print(f"  Full normalized: '{result.email}'")
    print(f"  ASCII representation: '{result.ascii_email}'")
    print(f"  Requires SMTPUTF8: {result.smtputf8}")

    # Show domain information if available
    if hasattr(result, 'domain_info') and result.domain_info:
        print(f"  Domain info available: Yes")

    print("Basic email validation and normalization demonstration completed successfully")

except Exception as e:
    print(f"Email validation demonstration completed with note: {e}")
    raise AssertionError(
        f"email_validator example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: Invalid Email Detection and Error Handling

In [ ]:
# Example 2: Invalid Email Detection and Error Handling

import email_validator
from email_validator import validate_email, EmailNotValidError

# Demonstrate invalid email detection and comprehensive error handling
print("=== Invalid Email Detection and Error Handling ===")

try:
    # List of intentionally invalid email addresses
    invalid_emails = [
        "",  # Empty string
        "plainaddress",  # No @ symbol
        "@domain.com",  # Missing local part
        "user@",  # Missing domain
        "user@@domain.com",  # Double @ symbol
        "user@domain",  # Missing TLD
        "user name@domain.com",  # Space in local part
        "user@domain .com",  # Space in domain
        "user@.domain.com",  # Domain starts with dot
        "user@domain..com",  # Double dot in domain
        "user@" + "a" * 250 + ".com",  # Domain too long
        "user@domain.123",  # All numeric TLD
        "user@[192.168.1.1",  # Malformed IP literal
        "user@domain.com.",  # Trailing dot
        "user..name@domain.com",  # Double dot in local part
        "user@-domain.com",  # Domain starts with hyphen
        "user@domain-.com"  # Domain ends with hyphen
    ]
    
    print("=== Invalid Email Testing ===")
    error_categories = {}
    
    # check_deliverability=False throughout this cell: these are SYNTAX rules, and deliverability is a live DNS
    # lookup (measured: with network, example.com is rejected for its null MX; offline it validates)
    for email in invalid_emails:
        try:
            result = validate_email(email, check_deliverability=False)
        except EmailNotValidError as e:
            error_message = str(e)
            
            # Categorize errors
            if "domain" in error_message.lower():
                category = "Domain Error"
            elif "local" in error_message.lower() or "part" in error_message.lower():
                category = "Local Part Error"
            elif "format" in error_message.lower() or "syntax" in error_message.lower():
                category = "Format Error"
            elif "length" in error_message.lower():
                category = "Length Error"
            else:
                category = "Other Error"
            
            error_categories[category] = error_categories.get(category, 0) + 1
            
            # Show first few examples of each category
            if error_categories[category] <= 3:
                display_email = email if email else "''" 
                print(f"✗ {display_email}")
                print(f"  Reason: {error_message}")
                print(f"  Category: {category}")
        else:
            raise AssertionError(f"validate_email accepted an invalid address {email!r} -> {result.normalized!r}")
    
    # Measured (email_validator 2.3.0): these two are ACCEPTED, legitimately. A one-letter TLD is syntactically
    # valid, and the 64-octet local-part limit (RFC 5321) is enforced only with strict=True.
    for email in ["user@domain.c", "a" * 65 + "@domain.com"]:
        print(f"✓ Syntactically valid by default: {validate_email(email, check_deliverability=False).normalized[:40]}")
    try:
        validate_email("a" * 65 + "@domain.com", check_deliverability=False, strict=True)
    except EmailNotValidError as e:
        print(f"✓ strict=True rejects a 65-character local part ({type(e).__name__})")
    else:
        raise AssertionError("strict=True accepted a 65-character local part")
    
    print(f"\n=== Error Summary ===")
    print(f"Total invalid emails tested: {len(invalid_emails)}")
    print(f"Error categories:")
    for category, count in error_categories.items():
        print(f"  {category}: {count}")
    
    # Detailed error analysis
    print(f"\n=== Detailed Error Analysis ===")
    
    error_test_cases = [
        ("", "Empty email address"),
        ("user@", "Missing domain part"),
        ("@domain.com", "Missing local part"),
        ("user@@domain.com", "Multiple @ symbols"),
        ("user name@domain.com", "Invalid character in local part"),
        ("user@domain..com", "Invalid domain format")
    ]
    
    for test_email, expected_issue in error_test_cases:
        try:
            validate_email(test_email, check_deliverability=False)
        except EmailNotValidError as e:
            print(f"✓ {expected_issue}: Correctly rejected")
            print(f"  Reason: {str(e)[:60]}...")
        else:
            raise AssertionError(f"{expected_issue}: validate_email accepted {test_email!r}")
    
    # Exception handling patterns
    print(f"\n=== Exception Handling Patterns ===")
    
    def safe_email_validation(email):
        """Safe email validation with detailed error reporting."""
        try:
            result = validate_email(email, check_deliverability=False)
            return {
                "valid": True,
                "normalized_email": result.email,
                "local_part": result.local_part,
                "domain_part": result.domain,
                "error": None
            }
        except EmailNotValidError as e:
            return {
                "valid": False,
                "normalized_email": None,
                "local_part": None,
                "domain_part": None,
                "error": str(e),
                "error_type": type(e).__name__
            }
        except Exception as e:
            return {
                "valid": False,
                "normalized_email": None,
                "local_part": None,
                "domain_part": None,
                "error": f"unexpected {type(e).__name__} ({e})",
                "error_type": type(e).__name__
            }
    
    # Test safe validation function: (input, expected error_type, or None when it must validate)
    test_cases = [
        ("valid@example.com", None),
        ("invalid@", "EmailSyntaxError"),
        ("user@@domain.com", "EmailSyntaxError"),
        (None, "TypeError")  # Test None input
    ]
    
    print("Safe validation function results:")
    for test_case, expected_error_type in test_cases:
        result = safe_email_validation(test_case)
        status = "✓ Valid" if result["valid"] else "✗ Invalid"
        test_display = str(test_case) if test_case is not None else "None"
        print(f"{status}: {test_display}")
        if not result["valid"]:
            print(f"  Reason: {result['error']}")
        if expected_error_type is None:
            assert result["valid"] and result["local_part"] == test_case.split("@")[0], f"{test_case!r} should validate: {result}"
        else:
            assert not result["valid"] and result["error_type"] == expected_error_type, f"{test_case!r} should fail with {expected_error_type}: {result}"

    print("Invalid email detection and error handling demonstration completed successfully")

except Exception as e:
    print(f"Error handling demonstration completed with note: {e}")
    raise AssertionError(
        f"email_validator example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: Bulk Email Validation and Performance Testing

In [ ]:
# Example 3: Bulk Email Validation and Performance Testing

import email_validator
from email_validator import validate_email, EmailNotValidError
import time

# Demonstrate bulk email validation and performance characteristics
print("=== Bulk Email Validation and Performance Testing ===")

try:
    # Generate test dataset for bulk validation
    print("=== Bulk Email Dataset Generation ===")
    
    # Create diverse email dataset
    email_patterns = [
        "user{0}@domain.com",
        "test.{0}@example.org", 
        "{0}+tag@service.net",
        "admin{0}@company.co.uk",
        "user{0}@subdomain.example.com"
    ]
    
    valid_emails = []
    invalid_emails = []
    
    # Generate valid emails
    for i in range(20):  # Generate 20 valid emails
        for pattern in email_patterns[:3]:  # Use first 3 patterns
            email = pattern.format(i)
            valid_emails.append(email)
    
    # Generate invalid emails
    invalid_patterns = [
        "invalid{0}@",  # Missing domain
        "@domain{0}.com",  # Missing local part
        "user{0}@@domain.com",  # Double @
        "user {0}@domain.com",  # Space in local
        "user{0}@domain..com"  # Double dot in domain
    ]
    
    for i in range(10):  # Generate 10 invalid emails
        for pattern in invalid_patterns[:2]:  # Use first 2 patterns
            email = pattern.format(i)
            invalid_emails.append(email)
    
    all_test_emails = valid_emails + invalid_emails
    
    print(f"Generated test dataset:")
    print(f"  Valid emails: {len(valid_emails)}")
    print(f"  Invalid emails: {len(invalid_emails)}")
    print(f"  Total emails: {len(all_test_emails)}")
    
    # Bulk validation function
    def bulk_validate_emails(email_list, batch_size=50):
        """Validate emails in batches with performance tracking."""
        results = {
            "valid": [],
            "invalid": [],
            "errors": [],
            "processing_time": 0,
            "emails_per_second": 0
        }
        
        start_time = time.time()
        
        for i in range(0, len(email_list), batch_size):
            batch = email_list[i:i + batch_size]
            
            for email in batch:
                try:
                    validation_result = validate_email(email, check_deliverability=False)  # syntax only; deliverability is a live DNS lookup
                    results["valid"].append({
                        "original": email,
                        "normalized": validation_result.email,
                        "domain": validation_result.domain
                    })
                except EmailNotValidError as e:
                    results["invalid"].append({
                        "email": email,
                        "error": str(e)
                    })
                except Exception as e:
                    results["errors"].append({
                        "email": email,
                        "error": f"Unexpected error: {str(e)}"
                    })
        
        end_time = time.time()
        results["processing_time"] = end_time - start_time
        
        total_processed = len(results["valid"]) + len(results["invalid"]) + len(results["errors"])
        if results["processing_time"] > 0:
            results["emails_per_second"] = total_processed / results["processing_time"]
        
        return results
    
    # Perform bulk validation
    print(f"\n=== Bulk Validation Results ===")
    
    bulk_results = bulk_validate_emails(all_test_emails[:50])  # Process first 50 emails
    
    print(f"Bulk validation completed:")
    print(f"  Processing time: {bulk_results['processing_time']:.3f} seconds")
    print(f"  Processing rate: {bulk_results['emails_per_second']:.1f} emails/second")
    print(f"  Valid emails: {len(bulk_results['valid'])}")
    print(f"  Invalid emails: {len(bulk_results['invalid'])}")
    print(f"  Processing errors: {len(bulk_results['errors'])}")
    
    # Show sample results
    print(f"\n=== Sample Valid Results ===")
    for result in bulk_results["valid"][:3]:  # Show first 3 valid results
        print(f"✓ {result['original']}")
        if result['original'] != result['normalized']:
            print(f"  Normalized: {result['normalized']}")
        print(f"  Domain: {result['domain']}")
    
    print(f"\n=== Sample Invalid Results ===")
    for result in bulk_results["invalid"][:3]:  # Show first 3 invalid results
        print(f"✗ {result['email']}")
        print(f"  Error: {result['error'][:50]}...")
    
    # Performance comparison
    print(f"\n=== Performance Comparison ===")
    
    test_sizes = [10, 25, 50]
    
    for size in test_sizes:
        test_subset = all_test_emails[:size]
        perf_results = bulk_validate_emails(test_subset)
        
        print(f"Dataset size {size}:")
        print(f"  Time: {perf_results['processing_time']:.3f}s")
        print(f"  Rate: {perf_results['emails_per_second']:.1f} emails/s")
        print(f"  Success rate: {len(perf_results['valid'])/size:.1%}")
    
    # Validation statistics
    print(f"\n=== Validation Statistics ===")
    
    domain_stats = {}
    for result in bulk_results["valid"]:
        domain = result["domain"]
        domain_stats[domain] = domain_stats.get(domain, 0) + 1
    
    print(f"Domain distribution (top 5):")
    sorted_domains = sorted(domain_stats.items(), key=lambda x: x[1], reverse=True)
    for domain, count in sorted_domains[:5]:
        print(f"  {domain}: {count} emails")

    print("Bulk email validation and performance testing completed successfully")

except Exception as e:
    print(f"Bulk validation demonstration completed with note: {e}")
    raise AssertionError(
        f"email_validator example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e